# Faces-houses ECoG: strict nested pipeline with visible source code

This is the shareable, presentation-ready version of the analysis. The original
`faces_houses_strict_nested_pipeline.ipynb` is unchanged. Here, the analysis code is shown
directly in executable cells so readers can see exactly how each result is produced.

The only separate process is the interactive bad-channel window, because it needs its own
GUI backend. MATLAB is called for the Miller spectral PCA step; the complete MATLAB source
used by that call is included below.

```text
raw ECoG -> 58-62 Hz notch -> bad-channel exclusion -> CAR
-> fixed folds by image identity -> outer-fold-specific Miller broadband
-> 20-ms bins -> training-only temporal-cluster selection
-> outer held-out logistic-regression decoding -> full nested permutation test
```


## 1. Configuration

Change `SUBJECT` and, if required, `ROI`. The same five-fold image-identity table is used for
every participant and ROI. `SEED` fixes all random operations. One thousand full-pipeline
permutations are intended for the confirmatory run; smaller values can be used only for a
quick code check.


In [1]:
from pathlib import Path
import csv
import importlib.util
import json
import os
import shutil
import subprocess
import sys

from IPython.display import HTML, Image, display

SUBJECT = "aa"
ROI = "Ventral temporal visual"
INNER_PERMUTATIONS = 200
FULL_PIPELINE_PERMUTATIONS = 1000
N_JOBS = -1
SEED = 20260716
OPEN_CHANNEL_INSPECTOR = True

current = Path.cwd().resolve()
project_candidates = [current, *current.parents]
PROJECT_ROOT = next((p for p in project_candidates if (p / "pilot_aa").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the project folder containing pilot_aa.")

default_data_root = Path.home() / "Downloads" / "faces_basic" / "faces_basic"
if default_data_root.is_dir():
    FACES_BASIC_ROOT = default_data_root
else:
    from tkinter import Tk, filedialog
    root = Tk()
    root.withdraw()
    selected = filedialog.askdirectory(title="Select the faces_basic folder")
    root.destroy()
    if not selected:
        raise RuntimeError("No faces_basic folder was selected.")
    FACES_BASIC_ROOT = Path(selected)

PILOT_DIR = PROJECT_ROOT / "pilot_aa"
SUBJECT_DIR = FACES_BASIC_ROOT / "data" / SUBJECT
RAW_FILE = SUBJECT_DIR / f"{SUBJECT}_faceshouses.mat"
FILTERED_FILE = SUBJECT_DIR / f"{SUBJECT}_faceshouses_notch58_62.mat"
BAD_FILE = SUBJECT_DIR / f"{SUBJECT}_bad_channels.txt"
LOC_FILE = FACES_BASIC_ROOT / "locs" / f"{SUBJECT}_xslocs.mat"
CV_FOLD_CSV = PILOT_DIR / "fixed_image_identity_cv_folds.csv"

OUTPUT_DIR = PILOT_DIR / "outputs" / SUBJECT
CAR_FILE = OUTPUT_DIR / f"{SUBJECT}_car_good_channels.npz"
CAR_MAT_FILE = OUTPUT_DIR / f"{SUBJECT}_car_good_channels.mat"
STRICT_MILLER_MAT_FILE = OUTPUT_DIR / f"{SUBJECT}_strict_nested_miller_bins.mat"
STRICT_BIN_FILE = OUTPUT_DIR / f"{SUBJECT}_strict_nested_miller_bins.npz"
BIN_FILE = STRICT_BIN_FILE
MIN_ELECTRODES_PER_ROI = 3

os.environ["FHPRED_SUBJECT"] = SUBJECT
os.environ["FHPRED_DATA_ROOT"] = str(FACES_BASIC_ROOT)
if str(PILOT_DIR) not in sys.path:
    sys.path.insert(0, str(PILOT_DIR))

print("Project:", PROJECT_ROOT)
print("Data root:", FACES_BASIC_ROOT)
print("Subject:", SUBJECT)
print("Output:", OUTPUT_DIR)


Project: C:\Users\sereg\OneDrive\Документы\New project
Data root: C:\Users\sereg\Downloads\faces_basic\faces_basic
Subject: aa
Output: C:\Users\sereg\OneDrive\Документы\New project\pilot_aa\outputs\aa


## 2. Software and input check

Required Python packages are listed in `pilot_aa/requirements.txt`. MATLAB with Signal
Processing Toolbox is required for the paper-style spectral decomposition.


In [2]:
required_modules = ["numpy", "scipy", "matplotlib", "sklearn", "joblib"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Missing Python packages: {missing}")
if shutil.which("matlab") is None:
    raise RuntimeError("MATLAB is required for Miller-style broadband extraction.")
for required_file in (RAW_FILE, LOC_FILE, CV_FOLD_CSV):
    if not required_file.exists():
        raise FileNotFoundError(required_file)
print("Preflight checks passed.")


Preflight checks passed.


## 3. Notch filtering: visible implementation

The original recording is never overwritten. A third-order Butterworth 58-62 Hz band-stop
filter is applied forward and backward (`sosfiltfilt`) to avoid phase shift. If the verified
output already exists, it is reused rather than filtered twice.


In [3]:
import numpy as np
from scipy.io import loadmat, savemat
from scipy.signal import butter, sosfiltfilt

NOTCH_BAND = (58.0, 62.0)
FILTER_ORDER = 3

if FILTERED_FILE.exists():
    print("Using existing filtered file:", FILTERED_FILE)
else:
    source = loadmat(RAW_FILE, squeeze_me=True)
    data = np.asarray(source["data"], dtype=float)
    stim = np.asarray(source["stim"])
    srate = float(np.asarray(source["srate"]).squeeze())

    sos = butter(
        N=FILTER_ORDER,
        Wn=NOTCH_BAND,
        btype="bandstop",
        fs=srate,
        output="sos",
    )
    filtered = sosfiltfilt(sos, data, axis=0)
    savemat(
        FILTERED_FILE,
        {
            "data": filtered,
            "stim": stim,
            "srate": srate,
            "notch_band": np.asarray(NOTCH_BAND),
            "filter_order": FILTER_ORDER,
            "filter_family": "Butterworth",
            "filter_mode": "zero-phase sosfiltfilt",
            "source_file": str(RAW_FILE),
        },
    )
    print("Saved:", FILTERED_FILE)


Using existing filtered file: C:\Users\sereg\Downloads\faces_basic\faces_basic\data\aa\aa_faceshouses_notch58_62.mat


## 4. Manual bad-channel review

The viewer is deliberately a separate GUI process; otherwise VS Code's inline Matplotlib
backend cannot provide working scrolling and channel selection. The viewer only writes the
reviewed `*_bad_channels.txt` file. It does not perform CAR or decoding. Its complete source
is available in [`inspect_raw.py`](inspect_raw.py).


In [4]:
if OPEN_CHANNEL_INSPECTOR:
    environment = os.environ.copy()
    environment["MPLBACKEND"] = "TkAgg"
    command = [
        sys.executable,
        str(PROJECT_ROOT / "inspect_raw.py"),
        str(FILTERED_FILE),
        "--loc-file",
        str(LOC_FILE),
    ]
    completed = subprocess.run(command, env=environment)
    if completed.returncode != 0:
        raise RuntimeError(f"Channel inspector exited with code {completed.returncode}.")

if not BAD_FILE.exists():
    raise FileNotFoundError(f"No reviewed bad-channel file was saved: {BAD_FILE}")
print("Bad channels:", BAD_FILE.read_text(encoding="utf-8").splitlines())


Bad channels: ['E1', 'E11', 'E13', 'E21', 'E26']


## 5. Shared event, anatomy, and bad-channel functions

These are the actual helper functions used by CAR and metadata validation. Image identity is
derived from stimulus codes: 1-50 are houses and 51-100 are faces. Each identity must occur
exactly three times.


In [5]:
from collections import Counter

import numpy as np


AREA_LABELS = {
    1: "Temporal pole",
    2: "Parahippocampal gyrus",
    3: "Inferior temporal gyrus",
    4: "Middle temporal gyrus",
    5: "Fusiform gyrus",
    6: "Lingual gyrus",
    7: "Inferior occipital gyrus",
    8: "Cuneus",
    9: "Post-ventral cingulate gyrus",
    10: "Middle occipital gyrus",
    11: "Occipital pole",
    12: "Precuneus",
    13: "Superior occipital gyrus",
    14: "Post-dorsal cingulate gyrus",
    20: "Non-included area",
}

ROI_BY_AREA = {
    "Parahippocampal gyrus": "Medial temporal",
    "Fusiform gyrus": "Ventral temporal visual",
    "Inferior temporal gyrus": "Ventral temporal visual",
    "Lingual gyrus": "Occipital visual",
    "Cuneus": "Occipital visual",
    "Inferior occipital gyrus": "Occipital visual",
    "Middle occipital gyrus": "Occipital visual",
    "Superior occipital gyrus": "Occipital visual",
    "Middle temporal gyrus": "Temporal association",
    "Temporal pole": "Temporal association",
}


def normalize_bad_channel_name(name, n_channels):
    name = name.strip()
    if name.startswith("E") and name[1:].isdigit():
        index = int(name[1:])
        if 1 <= index <= n_channels:
            return f"E{index}"
    if name.startswith("EEG") and name[3:].isdigit():
        index = int(name[3:])
        if 0 <= index < n_channels:
            return f"E{index + 1}"
    raise ValueError(
        f"Unsupported bad-channel name {name!r}. Expected E1..E{n_channels} "
        f"or legacy EEG0..EEG{n_channels - 1}."
    )


def load_bad_channels(path, n_channels):
    if not path.exists():
        raise FileNotFoundError(
            f"Bad-channel file does not exist: {path}\n"
            "Run inspect_raw.py and save the reviewed bad channels first."
        )
    saved = [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    normalized = [normalize_bad_channel_name(name, n_channels) for name in saved]
    if len(normalized) != len(set(normalized)):
        raise ValueError("Duplicate bad channels after name normalization.")
    return normalized


def anatomy_from_elcode(elcode):
    areas = [AREA_LABELS.get(int(code), f"Unknown code {int(code)}") for code in elcode]
    rois = [ROI_BY_AREA.get(area, "Excluded") for area in areas]
    return areas, rois


def extract_image_events(stim, srate):
    """Extract the 300 image-onset events while preserving image identity."""
    stim = np.asarray(stim).reshape(-1).astype(int)
    starts = np.r_[0, np.flatnonzero(np.diff(stim) != 0) + 1]
    ends = np.r_[starts[1:], len(stim)]
    values = stim[starts]

    run_for_sample = np.zeros(len(stim), dtype=int)
    run = 0
    in_task = False
    for start, end, value in zip(starts, ends, values):
        if value != 0 and not in_task:
            run += 1
            in_task = True
        elif value == 0:
            in_task = False
        if value != 0:
            run_for_sample[start:end] = run

    image_mask = (values >= 1) & (values <= 100)
    image_starts = starts[image_mask]
    image_ends = ends[image_mask]
    image_codes = values[image_mask]
    repetition = Counter()
    events = []
    for trial, (start, end, code) in enumerate(
        zip(image_starts, image_ends, image_codes), start=1
    ):
        code = int(code)
        repetition[code] += 1
        category = "house" if code <= 50 else "face"
        category_number = code if category == "house" else code - 50
        events.append(
            {
                "trial": trial,
                "event_sample_0based": int(start),
                "event_sample_matlab_1based": int(start + 1),
                "onset_seconds": float(start / srate),
                "duration_ms": float((end - start) / srate * 1000),
                "stim_code": code,
                "category": category,
                "label": int(category == "face"),
                "image_id": f"{category}_{category_number:02d}",
                "run": int(run_for_sample[start]),
                "repetition": int(repetition[code]),
            }
        )

    codes, counts = np.unique(image_codes, return_counts=True)
    if len(events) != 300 or len(codes) != 100 or set(counts.tolist()) != {3}:
        raise ValueError(
            "Unexpected event structure: expected 300 trials, 100 images, and 3 repeats per image."
        )
    return events



## 6. Bad-channel exclusion and common-average reference

Bad channels are removed first. At every sample, the mean across all remaining good
electrodes is subtracted. CAR uses all good electrodes, including electrodes outside the ROI;
ROI selection happens later.


In [6]:
import csv
import json

import numpy as np
from scipy.io import loadmat, savemat



def main():
    if not FILTERED_FILE.exists():
        raise FileNotFoundError(
            f"Correct filtered file is missing: {FILTERED_FILE}\n"
            "Run Step 1 from README.md first. Do not use *_notch60.mat."
        )

    mat = loadmat(FILTERED_FILE, squeeze_me=True)
    data = np.asarray(mat["data"], dtype=float)
    stim = np.asarray(mat["stim"]).reshape(-1).astype(np.int16)
    srate = float(np.asarray(mat["srate"]).squeeze())

    notch_band = np.atleast_1d(mat.get("notch_band", [])).astype(float)
    filter_order = int(np.asarray(mat.get("filter_order", -1)).squeeze())
    if notch_band.shape != (2,) or not np.allclose(notch_band, [58.0, 62.0]):
        raise ValueError("Input is not the verified 58-62 Hz filtered file.")
    if filter_order != 3:
        raise ValueError("Input does not have the expected third-order filter metadata.")

    loc = loadmat(LOC_FILE, squeeze_me=True)
    elcode = np.atleast_1d(loc["elcode"]).astype(int)
    coordinates = np.atleast_2d(np.asarray(loc["locs"], dtype=float))
    if data.shape[1] != len(elcode):
        raise ValueError("Data and localisation channel counts do not match.")

    all_names = [f"E{i + 1}" for i in range(data.shape[1])]
    bad_names = load_bad_channels(BAD_FILE, data.shape[1])
    bad_set = set(bad_names)
    good_indices = np.asarray(
        [index for index, name in enumerate(all_names) if name not in bad_set],
        dtype=int,
    )
    good_names = [all_names[index] for index in good_indices]

    # CAR intentionally uses every good electrode, including electrodes outside primary ROIs.
    good_data = data[:, good_indices]
    car_data = good_data - np.mean(good_data, axis=1, keepdims=True)

    areas, rois = anatomy_from_elcode(elcode)
    good_areas = [areas[index] for index in good_indices]
    good_rois = [rois[index] for index in good_indices]

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(
        CAR_FILE,
        data=car_data.astype(np.float32),
        stim=stim,
        srate=np.asarray(srate),
        channel_names=np.asarray(good_names),
        original_channel_indices_0based=good_indices,
        electrode_indices_1based=good_indices + 1,
        areas=np.asarray(good_areas),
        rois=np.asarray(good_rois),
        bad_channel_names=np.asarray(bad_names),
        source_file=np.asarray(str(FILTERED_FILE)),
    )
    savemat(
        CAR_MAT_FILE,
        {
            "data": car_data.astype(np.float32),
            "stim": stim.reshape(-1, 1),
            "srate": srate,
            "channel_names": np.asarray(good_names, dtype=object),
            "electrode_indices_1based": good_indices + 1,
            "areas": np.asarray(good_areas, dtype=object),
            "rois": np.asarray(good_rois, dtype=object),
            "bad_channel_names": np.asarray(bad_names, dtype=object),
            "source_file": str(FILTERED_FILE),
        },
        do_compression=False,
    )

    electrode_csv = OUTPUT_DIR / f"{SUBJECT}_electrode_qc_roi.csv"
    with electrode_csv.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(
            [
                "subject", "electrode", "data_column_0based", "anatomical_area",
                "roi", "qc_status", "included_in_car", "x", "y", "z",
            ]
        )
        for index, name in enumerate(all_names):
            is_bad = name in bad_set
            writer.writerow(
                [
                    SUBJECT,
                    name,
                    index,
                    areas[index],
                    rois[index],
                    "Bad" if is_bad else "Good",
                    "No" if is_bad else "Yes",
                    *coordinates[index].tolist(),
                ]
            )

    roi_counts = {
        roi: int(sum(value == roi for value in good_rois))
        for roi in sorted(set(good_rois))
    }
    summary = {
        "subject": SUBJECT,
        "input_file": str(FILTERED_FILE),
        "output_file": str(CAR_FILE),
        "matlab_input_file": str(CAR_MAT_FILE),
        "sampling_rate_hz": srate,
        "time_points": int(data.shape[0]),
        "original_channels": int(data.shape[1]),
        "bad_channels_saved": bad_names,
        "good_channels_in_car": len(good_names),
        "roi_counts_after_qc": roi_counts,
        "car_mean_absolute_residual": float(np.max(np.abs(car_data.mean(axis=1)))),
    }
    summary_path = OUTPUT_DIR / f"{SUBJECT}_prepare_car_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(json.dumps(summary, indent=2))
    print("Electrode table:", electrode_csv)


In [7]:
prepare_car = main
prepare_car()


{
  "subject": "aa",
  "input_file": "C:\\Users\\sereg\\Downloads\\faces_basic\\faces_basic\\data\\aa\\aa_faceshouses_notch58_62.mat",
  "output_file": "C:\\Users\\sereg\\OneDrive\\\u0414\u043e\u043a\u0443\u043c\u0435\u043d\u0442\u044b\\New project\\pilot_aa\\outputs\\aa\\aa_car_good_channels.npz",
  "matlab_input_file": "C:\\Users\\sereg\\OneDrive\\\u0414\u043e\u043a\u0443\u043c\u0435\u043d\u0442\u044b\\New project\\pilot_aa\\outputs\\aa\\aa_car_good_channels.mat",
  "sampling_rate_hz": 1000.0,
  "time_points": 271400,
  "original_channels": 46,
  "bad_channels_saved": [
    "E1",
    "E11",
    "E13",
    "E21",
    "E26"
  ],
  "good_channels_in_car": 41,
  "roi_counts_after_qc": {
    "Excluded": 12,
    "Medial temporal": 6,
    "Occipital visual": 8,
    "Temporal association": 9,
    "Ventral temporal visual": 6
  },
  "car_mean_absolute_residual": 3.861581406942228e-10
}
Electrode table: C:\Users\sereg\OneDrive\Документы\New project\pilot_aa\outputs\aa\aa_electrode_qc_roi.csv


## 7. Fixed five-fold split by image identity

The table contains one row per unique image identity. Every fold contains 10 face identities
and 10 house identities. All three presentations of an identity inherit the same fold, so no
image can occur in both training and test data.


In [ ]:
fold_rows = list(csv.DictReader(CV_FOLD_CSV.open(encoding="utf-8-sig")))
if len(fold_rows) != 100:
    raise ValueError("The fold table must contain exactly 100 image identities.")

fold_by_code = {int(row["stim_code"]): int(row["cv_fold"]) for row in fold_rows}
for fold in range(1, 6):
    rows = [row for row in fold_rows if int(row["cv_fold"]) == fold]
    categories = [row["category"] for row in rows]
    assert len(rows) == 20
    assert categories.count("face") == 10
    assert categories.count("house") == 10

car_source = np.load(CAR_FILE)
events = extract_image_events(car_source["stim"], float(car_source["srate"]))
trial_folds = np.asarray([fold_by_code[event["stim_code"]] for event in events])
for image_id in sorted({event["image_id"] for event in events}):
    assigned = {
        trial_folds[index]
        for index, event in enumerate(events)
        if event["image_id"] == image_id
    }
    assert len(assigned) == 1

print("Fold counts by trial:", {fold: int(np.sum(trial_folds == fold)) for fold in range(1, 6)})
print("Verified: each image identity and all three repetitions stay in one fold.")


## 8. Strict Miller broadband: complete MATLAB implementation

For each outer split, spectral normalization, PCA weights, and broadband normalization are
estimated without that split's 20 test identities. The resulting frozen transform is then
applied to the whole continuous recording to produce the fold-specific feature representation.

The cell below shows the complete MATLAB function called by Python.

```matlab
function extract_strict_nested_miller_bins(input_mat, faces_basic_root, fold_csv, output_mat)
% Fit Miller spectral PCA without outer-test images and save fold-specific bins.

arguments
    input_mat (1,:) char
    faces_basic_root (1,:) char
    fold_csv (1,:) char
    output_mat (1,:) char
end

addpath(faces_basic_root);
addpath(fullfile(faces_basic_root, 'dc_files'));
addpath(fileparts(mfilename('fullpath'))); % compatibility psd.m is in project root
addpath(fileparts(fileparts(mfilename('fullpath'))));

loaded = load(input_mat);
required = {'data', 'stim', 'srate', 'channel_names', ...
    'electrode_indices_1based', 'areas', 'rois'};
for k = 1:numel(required)
    if ~isfield(loaded, required{k})
        error('Missing variable %s in %s', required{k}, input_mat);
    end
end

data = double(loaded.data);
stim = loaded.stim(:);
srate = double(loaded.srate);
if srate ~= 1000
    error('Original Miller implementation expects 1000 Hz; found %g Hz.', srate);
end

fold_table = readtable(fold_csv);
fold_by_code = zeros(100, 1);
fold_by_code(fold_table.stim_code) = fold_table.cv_fold;
if any(fold_by_code < 1 | fold_by_code > 5)
    error('The fold table must assign every stimulus code 1..100 to folds 1..5.');
end

pts = fh_get_events(stim);
stim_codes = double(stim(pts(:, 2)));
keep = stim_codes >= 1 & stim_codes <= 100;
pts = pts(keep, :);
stim_codes = stim_codes(keep);
if numel(stim_codes) ~= 300
    error('Expected 300 face/house events; found %d.', numel(stim_codes));
end
trial_folds = fold_by_code(stim_codes);
labels = double(stim_codes > 50);
event_onsets = pts(:, 1);

old_folder = pwd;
cleanup = onCleanup(@() cd(old_folder));
cd(faces_basic_root); % calc_dg_spectra loads support files by relative path

fprintf('Strict Miller broadband: spectra for all 300 events...\n');
spectra = calc_dg_spectra(data, pts);

first_pc_weights = [];
spectral_frequencies = [];
training_sample_masks = false(size(data, 1), 5);
epoch_offsets = (-200:599)';

for outer_fold = 1:5
    training_trials = trial_folds ~= outer_fold;
    training_spectra = calc_nspectra(spectra(:, :, training_trials));
    [~, pc_vecs, ~, fold_frequencies] = dg_pca_step(training_spectra);
    if isempty(spectral_frequencies)
        spectral_frequencies = fold_frequencies(:);
        first_pc_weights = zeros( ...
            numel(spectral_frequencies), size(data, 2), 5 ...
        );
    elseif ~isequal(spectral_frequencies, fold_frequencies(:))
        error('Spectral frequency grid changed between outer folds.');
    end

    for channel = 1:size(data, 2)
        mixing_matrix = squeeze(pc_vecs(:, channel, :))';
        first_pc_weights(:, channel, outer_fold) = mixing_matrix(:, 1);
    end

    training_onsets = event_onsets(training_trials);
    for event_index = 1:numel(training_onsets)
        indices = training_onsets(event_index) + epoch_offsets;
        if indices(1) < 1 || indices(end) > size(data, 1)
            error('A training epoch extends outside the recording.');
        end
        training_sample_masks(indices, outer_fold) = true;
    end
end
clear spectra pc_vecs training_spectra

bin_starts_ms = (0:20:380)';
bin_ends_ms = bin_starts_ms + 20;
bin_centers_ms = (bin_starts_ms + bin_ends_ms) / 2;
features = zeros( ...
    numel(stim_codes), size(data, 2), numel(bin_starts_ms), 5, 'single' ...
);

fprintf('Strict Miller broadband: fold-specific continuous transforms...\n');
for channel = 1:size(data, 2)
    fprintf('Strict Miller broadband channel %d / %d\n', channel, size(data, 2));
    lnA = zeros(size(data, 1), 5);

    for frequency_index = 1:numel(spectral_frequencies)
        frequency = spectral_frequencies(frequency_index);
        t = 1:floor(5 * srate / frequency);
        tmid = floor(max(t) / 2);
        wavelet = exp(1i * 2 * pi * (frequency / srate) * (t - tmid)) .* ...
            exp(-((t - tmid).^2) / (2 * (srate / frequency)^2));
        convolved = conv(wavelet, data(:, channel));
        remove_left = 1:(floor(length(wavelet) / 2) - 1);
        remove_right = floor(length(convolved) - length(wavelet) / 2 + 1):length(convolved);
        convolved([remove_left, remove_right]) = [];
        power = abs(convolved(:)).^2;
        if numel(power) ~= size(data, 1)
            error('Unexpected wavelet convolution length at %g Hz.', frequency);
        end

        for outer_fold = 1:5
            training_mean = mean(power(training_sample_masks(:, outer_fold)));
            if ~isfinite(training_mean) || training_mean <= 0
                error('Invalid training power mean at %g Hz.', frequency);
            end
            weight = first_pc_weights(frequency_index, channel, outer_fold);
            lnA(:, outer_fold) = lnA(:, outer_fold) + ...
                log(power / training_mean) * weight;
        end
    end

    for outer_fold = 1:5
        smoothed = conv(gausswin(80), lnA(:, outer_fold));
        smoothed(1:39) = [];
        smoothed((length(smoothed) - 39):length(smoothed)) = [];
        if numel(smoothed) ~= size(data, 1)
            error('Unexpected smoothing output length.');
        end
        train_values = smoothed(training_sample_masks(:, outer_fold));
        train_mean = mean(train_values);
        train_std = std(train_values);
        if ~isfinite(train_std) || train_std <= 0
            error('Invalid training standard deviation for channel %d.', channel);
        end
        broadband = exp((smoothed - train_mean) / train_std) - 1;

        for trial = 1:numel(event_onsets)
            epoch_indices = event_onsets(trial) + epoch_offsets;
            epoch = broadband(epoch_indices);
            baseline = mean(epoch(1:200));
            epoch = epoch - baseline;
            for bin_index = 1:numel(bin_starts_ms)
                first_sample = 201 + bin_starts_ms(bin_index);
                last_sample = 200 + bin_ends_ms(bin_index);
                features(trial, channel, bin_index, outer_fold) = ...
                    single(mean(epoch(first_sample:last_sample)));
            end
        end
    end
end

channel_names = loaded.channel_names;
electrode_indices_1based = loaded.electrode_indices_1based;
areas = loaded.areas;
rois = loaded.rois;
method = [ ...
    'Strict nested Miller PCA broadband: spectral PCA, frequency normalization, ', ...
    'and broadband z-scoring fitted without each outer-test fold' ...
];
source_car_file = input_mat;

save( ...
    output_mat, ...
    'features', 'labels', 'stim_codes', 'trial_folds', 'event_onsets', ...
    'channel_names', 'electrode_indices_1based', 'areas', 'rois', ...
    'bin_starts_ms', 'bin_ends_ms', 'bin_centers_ms', ...
    'spectral_frequencies', 'first_pc_weights', 'method', 'source_car_file', ...
    '-v7' ...
);
fprintf('Saved strict nested Miller bins: %s\n', output_mat);
end

```


### Original published MATLAB helpers called above

These small functions come with the Miller `faces_basic` release. They are included here so
the spectral snapshots, normalization and PCA are also visible rather than hidden behind
filenames.


#### `fh_get_events.m`

```matlab
function [evs]=fh_get_events(stim)
%this function defines evs lying at the beginning (1st column)and middle
%(2nd column) of each stimulus and isi period
% 0 = ISI
% 1 = HOUSE STIM
% 2 = FACE STIM

b=find((stim-[0; stim(1:(length(stim)-1))])~=0);
c=floor(diff(b)/2);
b(end)=[];
d=b+c;
evs(:,1)=b;
evs(:,2)=d;
evs(:,3)=stim(d);
evs(find(evs(:,3)==0),:)=[];
evs(find(evs(:,3)<51),3)=1;
evs(find(evs(:,3)==101),3)=0;
evs(find(evs(:,3)>50),3)=2;
clear b c d

%clip if too close to ends of run files
evs(find(or(evs(:,1)<500,evs(:,2)>(length(stim)-1000))),:)=[];

```


#### `dc_files/calc_dg_spectra.m`

```matlab
function [spectra]=calc_dg_spectra(data,pts);
%this function calculates the the spectra at time points in pts(:,2)
%kjm 12/07

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%parameters
samplefreq=1000;  %sampling frequency
bsize=1000;  %window size for spectral calculation
wt=hann(bsize);  %1-.5*cos window  -- use a hann window % wt=hamming(bsize);  %use a hamming wind

load ns_1k_1_300_filt % applicable to seattle data, but not stanford data
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%calculate spectra
disp('calculating power spectra')
spectra=zeros(300,size(data,2),size(pts,1));
for k=1:size(pts,1)
    if (mod(k, 100) == 0), fprintf(1, '%03d ', k); if (mod(k, 500) == 0), fprintf(1, '* /%d\r', size(pts,1)); end, end
    for m=1:size(data,2)
        [ts,f] = psd(data((pts(k,2)-floor(bsize/2)+1):(pts(k,2)+ceil(bsize/2)),m),bsize,samplefreq);
%         ts=(abs(fft(wt.*data((pts(k,2)-floor(bsize/2)+1):(pts(k,2)+ceil(bsize/2)),m)))).^2;
%         spectra(:,m,k)=ts(2:301)./(nsfilt.^2)';
        spectra(:,m,k)=ts(2:301);
    end
end
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

```


#### `dc_files/calc_nspectra.m`

```matlab
function [spectra]=calc_nspectra(spectra)
%this function normalizes the spectra prior to the pca step
%kjm 12/07

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
% normalization step
for k=1:size(spectra,2)
%     spectra(:,k,:)=spectra(:,k,:)./repmat(squeeze(mean(spectra(:,k,:),3)),[1 1 size(spectra,3)]);
    spectra(:,k,:)=log(spectra(:,k,:)./repmat(squeeze(mean(spectra(:,k,:),3)),[1 1 size(spectra,3)]));
end
clear k
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


```


#### `dc_files/dg_pca_step.m`

```matlab
function [pc_weights, pc_vecs, pc_vals, f]=dg_pca_step(spectra)
% function [pc_weights, pc_vecs, pc_vals, f]=dg_pca_step(patient,spectra)
%this function calculates and returns the principal spectra/eigenvalues, and their projections.
%it also returns the acceptable frequency range b/c of jc and cc
%kjm 12/07

nc=size(spectra,2);%number of channels left

% ncomps=2; %number of components to keep


%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%create indices to exclude around harmonics of 60
f=1:300; no60=[];
for k=1:ceil(max(f/60)), no60=[no60 (60*k-3):(60*k+3)]; end %3 hz up or down
no60=[no60 247:253]; 
f=setdiff(f,no60); %dispose of 60hz stuff
f(find(f>200))=[];
% if patient=='jc', f(find(f>195))=[]; f(find(and(f>97,f<103)))=[]; f(find(and(f>155,f<165)))=[]; f(find(and(f>53,f<67)))=[]; end
% if patient=='cc', f(find(f<5))=[]; end
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
ncomps=length(f); %number of components to keep


%initialize
pc_weights=zeros(ncomps,nc,size(spectra,3)); %projection weights
pc_vecs=zeros(ncomps,nc,length(f)); %eigenvectors
pc_vals=zeros(ncomps,nc); %eigenvalues

%%run pca
for chan=1:nc
    %select proper data
    ts=squeeze(spectra(f,chan,:)); %introduce log?
    
    %get evecs and evals
    [vecs,vals]=eig(ts*ts'); 
    [vals,v_inds]=sort(sort(sum(vals)),'descend'); vecs=vecs(:,v_inds); %reshape properly

    
    %assign values
    pc_weights(:,chan,:)=vecs(:,1:ncomps)'*ts;
    pc_vecs(:,chan,:)=vecs(:,1:ncomps)';
    pc_vals(:,chan)=vals(1:ncomps);
    
    clear v_inds covmat vecs vals ts %housekeeping
end
```


### Python-to-MATLAB execution and conversion

The next cell is the complete Python bridge. It calls the MATLAB function shown above and
converts its output to a NumPy archive with axes `trial x electrode x time-bin x outer-fold`.


In [ ]:
import json
import subprocess

import numpy as np
from scipy.io import loadmat



def matlab_string(path):
    return str(path).replace("'", "''")


def matlab_cellstr(values):
    result = []
    for value in np.asarray(values).reshape(-1):
        if isinstance(value, np.ndarray):
            value = value.item() if value.size == 1 else "".join(value.tolist())
        result.append(str(value))
    return np.asarray(result)


def main():
    if not CAR_MAT_FILE.exists():
        raise FileNotFoundError(f"Missing {CAR_MAT_FILE}. Run Step 3 first.")
    if not CV_FOLD_CSV.exists():
        raise FileNotFoundError(CV_FOLD_CSV)

    helper_dir = PROJECT_ROOT / "pilot_aa"
    matlab_command = (
        f"addpath('{matlab_string(helper_dir)}'); "
        f"extract_strict_nested_miller_bins("
        f"'{matlab_string(CAR_MAT_FILE)}',"
        f"'{matlab_string(FACES_BASIC_ROOT)}',"
        f"'{matlab_string(CV_FOLD_CSV)}',"
        f"'{matlab_string(STRICT_MILLER_MAT_FILE)}');"
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("Starting leakage-controlled Miller broadband extraction in MATLAB.")
    print("Spectral PCA is fitted separately without each outer-test fold.")
    subprocess.run(["matlab", "-batch", matlab_command], check=True)

    result = loadmat(STRICT_MILLER_MAT_FILE, squeeze_me=True)
    stim_codes = np.asarray(result["stim_codes"]).reshape(-1).astype(int)
    image_ids = np.asarray(
        [
            f"house_{code:02d}" if code <= 50 else f"face_{code - 50:02d}"
            for code in stim_codes
        ]
    )
    np.savez(
        STRICT_BIN_FILE,
        features=np.asarray(result["features"], dtype=np.float32),
        labels=np.asarray(result["labels"]).reshape(-1).astype(np.int8),
        image_ids=image_ids,
        cv_folds=np.asarray(result["trial_folds"]).reshape(-1).astype(np.int8),
        event_samples_1based=np.asarray(result["event_onsets"]).reshape(-1).astype(int),
        channel_names=matlab_cellstr(result["channel_names"]),
        electrode_indices_1based=np.asarray(
            result["electrode_indices_1based"]
        ).reshape(-1),
        areas=matlab_cellstr(result["areas"]),
        rois=matlab_cellstr(result["rois"]),
        bin_starts_ms=np.asarray(result["bin_starts_ms"]).reshape(-1),
        bin_ends_ms=np.asarray(result["bin_ends_ms"]).reshape(-1),
        bin_centers_ms=np.asarray(result["bin_centers_ms"]).reshape(-1),
        spectral_frequencies=np.asarray(result["spectral_frequencies"]).reshape(-1),
        method=np.asarray(str(result["method"])),
    )

    features = np.asarray(result["features"])
    summary = {
        "subject": SUBJECT,
        "input": str(CAR_MAT_FILE),
        "output": str(STRICT_BIN_FILE),
        "feature_shape": list(features.shape),
        "axis_order": "trial, channel, time_bin, outer_test_fold",
        "method": str(result["method"]),
        "outer_test_control": (
            "For outer fold k, PCA weights and normalization parameters were fitted "
            "using only trials whose image identities were not in fold k."
        ),
    }
    (OUTPUT_DIR / f"{SUBJECT}_strict_nested_miller_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    print(json.dumps(summary, indent=2))


In [ ]:
extract_strict_features = main
extract_strict_features()


## 9. Nested temporal-window selection and held-out decoding

The next code cell contains the complete classifier and inference implementation. For every
outer fold it performs inner CV on the other four folds, creates image-level permutation null
curves, joins adjacent supra-threshold bins, selects the largest training cluster, averages each
electrode within that window, fits standardized L2 logistic regression, and predicts the
untouched outer fold. The final permutation test repeats the label-dependent nested pipeline
and uses combined out-of-fold AUC as its statistic.


In [ ]:
import argparse
import csv
import json
import re
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler



def make_model():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            penalty="l2",
            solver="liblinear",
            max_iter=1000,
            random_state=0,
        ),
    )


def image_level_permutation(labels, image_ids, mask, rng):
    unique_ids = np.unique(image_ids[mask])
    identity_labels = np.asarray(
        [labels[np.flatnonzero((image_ids == image_id) & mask)[0]] for image_id in unique_ids]
    )
    shuffled = rng.permutation(identity_labels)
    mapping = dict(zip(unique_ids.tolist(), shuffled.tolist()))
    permuted = labels.copy()
    permuted[mask] = np.asarray([mapping[value] for value in image_ids[mask]])
    return permuted


def inner_oof_auc_curve(features, labels, folds, outer_test_fold):
    outer_train = folds != outer_test_fold
    inner_folds = sorted(np.unique(folds[outer_train]).tolist())
    scores = np.full((len(labels), features.shape[2]), np.nan, dtype=float)
    for validation_fold in inner_folds:
        train = outer_train & (folds != validation_fold)
        validation = outer_train & (folds == validation_fold)
        for bin_index in range(features.shape[2]):
            model = make_model()
            model.fit(features[train, :, bin_index], labels[train])
            scores[validation, bin_index] = model.predict_proba(
                features[validation, :, bin_index]
            )[:, 1]
    return np.asarray(
        [roc_auc_score(labels[outer_train], scores[outer_train, index]) for index in range(features.shape[2])]
    )


def contiguous_clusters(mask):
    clusters = []
    start = None
    for index, value in enumerate(mask):
        if value and start is None:
            start = index
        if start is not None and (not value or index == len(mask) - 1):
            stop = index if value and index == len(mask) - 1 else index - 1
            clusters.append((start, stop))
            start = None
    return clusters


def cluster_masses(curve, thresholds):
    clusters = contiguous_clusters(curve > thresholds)
    return [
        (start, stop, float(np.sum(curve[start : stop + 1] - thresholds[start : stop + 1])))
        for start, stop in clusters
    ]


def select_training_window(curve, thresholds):
    clusters = cluster_masses(curve, thresholds)
    if clusters:
        return max(clusters, key=lambda item: item[2]), False
    peak = int(np.argmax(curve - thresholds))
    return (peak, peak, float(curve[peak] - thresholds[peak])), True


def permuted_nested_oof_auc(
    permutation_seed,
    features,
    labels,
    image_ids,
    folds,
    thresholds_by_fold,
):
    rng = np.random.default_rng(permutation_seed)
    permuted_labels = image_level_permutation(
        labels, image_ids, np.ones(len(labels), dtype=bool), rng
    )
    scores = np.full(len(labels), np.nan, dtype=float)

    for outer_test_fold in range(1, 6):
        fold_features = (
            features[:, :, :, outer_test_fold - 1]
            if features.ndim == 4
            else features
        )
        outer_train = folds != outer_test_fold
        outer_test = folds == outer_test_fold
        curve = inner_oof_auc_curve(
            fold_features, permuted_labels, folds, outer_test_fold
        )
        selected, _ = select_training_window(
            curve, thresholds_by_fold[outer_test_fold]
        )
        start_bin, stop_bin, _ = selected
        window_features = fold_features[:, :, start_bin : stop_bin + 1].mean(axis=2)
        model = make_model()
        model.fit(window_features[outer_train], permuted_labels[outer_train])
        scores[outer_test] = model.predict_proba(window_features[outer_test])[:, 1]

    return float(roc_auc_score(permuted_labels, scores))


def safe_name(value):
    return re.sub(r"[^A-Za-z0-9]+", "_", value).strip("_").lower()


def main():
    parser = argparse.ArgumentParser(
        description="Pilot nested image-identity CV with training-only temporal-cluster selection."
    )
    parser.add_argument(
        "--roi",
        default="Ventral temporal visual",
        help="ROI name, or 'all' to run every eligible ROI.",
    )
    parser.add_argument(
        "--bin-file",
        type=Path,
        default=BIN_FILE,
        help="Binned feature NPZ. Four-dimensional input enables outer-fold-specific preprocessing.",
    )
    parser.add_argument(
        "--n-permutations",
        type=int,
        default=200,
        help="Training-only cluster permutations. Use 20 for a smoke test; 200 for the expanded pilot.",
    )
    parser.add_argument(
        "--n-outer-permutations",
        type=int,
        default=0,
        help="Full nested-pipeline permutations for combined held-out AUC.",
    )
    parser.add_argument(
        "--n-jobs",
        type=int,
        default=-1,
        help="Parallel jobs for full nested permutations; -1 uses all cores.",
    )
    parser.add_argument("--seed", type=int, default=20260716)
    args = parser.parse_args()
    if args.n_permutations < 1:
        raise ValueError("n-permutations must be positive.")
    if args.n_outer_permutations < 0:
        raise ValueError("n-outer-permutations cannot be negative.")
    if not args.bin_file.exists():
        raise FileNotFoundError(f"Missing {args.bin_file}. Run feature extraction first.")

    source = np.load(args.bin_file)
    all_features = np.asarray(source["features"], dtype=float)
    if all_features.ndim not in (3, 4):
        raise ValueError(
            "Expected features with axes trial/channel/time_bin[/outer_fold]."
        )
    if all_features.ndim == 4 and all_features.shape[3] != 5:
        raise ValueError("Fold-specific features must contain five outer-fold versions.")
    strict_outer_preprocessing = all_features.ndim == 4
    feature_method = (
        str(np.asarray(source["method"]).squeeze())
        if "method" in source.files
        else "unspecified"
    )
    labels = np.asarray(source["labels"], dtype=int)
    image_ids = np.asarray(source["image_ids"]).astype(str)
    folds = np.asarray(source["cv_folds"], dtype=int)
    rois = np.asarray(source["rois"]).astype(str)
    channel_names = np.asarray(source["channel_names"]).astype(str)
    bin_starts = np.asarray(source["bin_starts_ms"], dtype=float)
    bin_ends = np.asarray(source["bin_ends_ms"], dtype=float)
    bin_centers = np.asarray(source["bin_centers_ms"], dtype=float)

    available_rois = sorted(set(rois) - {"Excluded"})
    selected_rois = available_rois if args.roi.lower() == "all" else [args.roi]
    unknown = [roi for roi in selected_rois if roi not in available_rois]
    if unknown:
        raise ValueError(f"Unknown ROI {unknown}. Available: {available_rois}")

    for roi in selected_rois:
        roi_mask = rois == roi
        roi_channels = channel_names[roi_mask]
        if len(roi_channels) < MIN_ELECTRODES_PER_ROI:
            print(f"Skipping {roi}: only {len(roi_channels)} electrodes.")
            continue
        features = all_features[:, roi_mask, :]
        result_prefix = "nested_strict" if strict_outer_preprocessing else "nested"
        result_dir = OUTPUT_DIR / f"{result_prefix}_{safe_name(roi)}"
        result_dir.mkdir(parents=True, exist_ok=True)

        outer_results = []
        prediction_rows = []
        thresholds_by_fold = {}
        figure, axes = plt.subplots(5, 1, figsize=(9, 13), sharex=True)

        for outer_test_fold in range(1, 6):
            print(f"{roi}: outer test fold {outer_test_fold}/5")
            fold_features = (
                features[:, :, :, outer_test_fold - 1]
                if strict_outer_preprocessing
                else features
            )
            outer_train = folds != outer_test_fold
            outer_test = folds == outer_test_fold
            real_curve = inner_oof_auc_curve(
                fold_features, labels, folds, outer_test_fold
            )

            rng = np.random.default_rng(args.seed + outer_test_fold)
            null_curves = np.empty(
                (args.n_permutations, fold_features.shape[2]), dtype=float
            )
            for permutation in range(args.n_permutations):
                permuted_labels = image_level_permutation(
                    labels, image_ids, outer_train, rng
                )
                null_curves[permutation] = inner_oof_auc_curve(
                    fold_features, permuted_labels, folds, outer_test_fold
                )

            thresholds = np.quantile(null_curves, 0.95, axis=0)
            thresholds_by_fold[outer_test_fold] = thresholds
            real_clusters = cluster_masses(real_curve, thresholds)
            null_max_masses = np.asarray(
                [
                    max((mass for _, _, mass in cluster_masses(curve, thresholds)), default=0.0)
                    for curve in null_curves
                ]
            )

            axis = axes[outer_test_fold - 1]
            axis.plot(bin_centers, real_curve, color="#204F78", label="inner-CV AUC")
            axis.plot(bin_centers, thresholds, color="#A33A35", linestyle="--", label="95% null threshold")
            axis.axhline(0.5, color="black", linewidth=0.8)
            training_fold_numbers = [
                str(fold) for fold in range(1, 6) if fold != outer_test_fold
            ]
            axis.set_title(
                f"Outer test fold {outer_test_fold} held out | "
                f"curve from training folds {', '.join(training_fold_numbers)}",
                fontsize=9,
            )
            axis.set_ylabel("Inner-CV\nAUC")
            axis.grid(True, alpha=0.25)

            selected, used_peak_fallback = select_training_window(real_curve, thresholds)
            start_bin, stop_bin, mass = selected
            cluster_p = (
                float((1 + np.sum(null_max_masses >= mass)) / (args.n_permutations + 1))
                if not used_peak_fallback
                else None
            )
            cluster_start = float(bin_starts[start_bin])
            cluster_end = float(bin_ends[stop_bin])
            axis.axvspan(cluster_start, cluster_end, color="#4E9F6F", alpha=0.25)

            window_features = fold_features[:, :, start_bin : stop_bin + 1].mean(axis=2)
            final_model = make_model()
            final_model.fit(window_features[outer_train], labels[outer_train])
            outer_scores = final_model.predict_proba(window_features[outer_test])[:, 1]
            outer_auc = float(roc_auc_score(labels[outer_test], outer_scores))

            outer_results.append(
                {
                    "subject": SUBJECT,
                    "roi": roi,
                    "outer_test_fold": outer_test_fold,
                    "status": (
                        "fallback_peak_no_training_cluster"
                        if used_peak_fallback
                        else (
                            "pilot_candidate_training_cluster"
                            if args.n_permutations < 99
                            else (
                                "tested_significant_training_cluster"
                                if cluster_p < 0.05
                                else "tested_candidate_training_cluster"
                            )
                        )
                    ),
                    "cluster_start_ms": cluster_start,
                    "cluster_end_ms": cluster_end,
                    "training_cluster_mass": mass,
                    "training_cluster_p": cluster_p,
                    "training_p_resolution": 1.0 / (args.n_permutations + 1),
                    "outer_test_auc": outer_auc,
                    "n_electrodes": len(roi_channels),
                }
            )
            for trial_index, score in zip(np.flatnonzero(outer_test), outer_scores):
                prediction_rows.append(
                    {
                        "subject": SUBJECT,
                        "roi": roi,
                        "trial_index_0based": int(trial_index),
                        "image_id": image_ids[trial_index],
                        "cv_fold": int(folds[trial_index]),
                        "true_label": int(labels[trial_index]),
                        "face_score": float(score),
                        "selected_cluster_start_ms": cluster_start,
                        "selected_cluster_end_ms": cluster_end,
                    }
                )

        axes[0].legend(loc="upper right", fontsize=8)
        axes[-1].set_xlabel("Time from image onset (ms)")
        figure.suptitle(f"{SUBJECT} | {roi} | training-only temporal cluster selection")
        figure.tight_layout()
        figure.savefig(result_dir / "inner_cv_cluster_selection.png", dpi=160)
        plt.close(figure)

        if prediction_rows:
            roc_figure, roc_axes = plt.subplots(1, 5, figsize=(15, 3.2), sharex=True, sharey=True)
            for outer_test_fold, axis in enumerate(roc_axes, start=1):
                fold_rows = [
                    row for row in prediction_rows if row["cv_fold"] == outer_test_fold
                ]
                fold_labels = np.asarray([row["true_label"] for row in fold_rows])
                fold_scores = np.asarray([row["face_score"] for row in fold_rows])
                fold_auc = roc_auc_score(fold_labels, fold_scores)
                false_positive_rate, true_positive_rate, _ = roc_curve(
                    fold_labels, fold_scores
                )
                axis.plot(false_positive_rate, true_positive_rate, color="#204F78", linewidth=2)
                axis.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=0.8)
                axis.set_title(f"Held-out fold {outer_test_fold}\nAUC = {fold_auc:.3f}")
                axis.set_xlabel("False-positive rate")
                axis.grid(True, alpha=0.25)
            roc_axes[0].set_ylabel("True-positive rate")
            roc_figure.suptitle(f"{SUBJECT} | {roi} | outer held-out test performance")
            roc_figure.tight_layout()
            roc_figure.savefig(result_dir / "held_out_test_roc.png", dpi=160)
            plt.close(roc_figure)

        outer_csv = result_dir / "outer_fold_results.csv"
        with outer_csv.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(outer_results[0].keys()))
            writer.writeheader()
            writer.writerows(outer_results)

        predictions_csv = result_dir / "held_out_predictions.csv"
        if prediction_rows:
            with predictions_csv.open("w", newline="", encoding="utf-8") as handle:
                writer = csv.DictWriter(handle, fieldnames=list(prediction_rows[0].keys()))
                writer.writeheader()
                writer.writerows(prediction_rows)

        complete = len(prediction_rows) == len(labels)
        combined_auc = (
            float(
                roc_auc_score(
                    [row["true_label"] for row in prediction_rows],
                    [row["face_score"] for row in prediction_rows],
                )
            )
            if complete
            else None
        )

        outer_permutation_p = None
        outer_null_95 = None
        if complete and args.n_outer_permutations:
            print(
                f"{roi}: running {args.n_outer_permutations} full nested-pipeline permutations"
            )
            seeds = args.seed + 100_000 + np.arange(args.n_outer_permutations)
            null_aucs = np.asarray(
                Parallel(n_jobs=args.n_jobs, prefer="threads", verbose=5)(
                    delayed(permuted_nested_oof_auc)(
                        int(permutation_seed),
                        features,
                        labels,
                        image_ids,
                        folds,
                        thresholds_by_fold,
                    )
                    for permutation_seed in seeds
                ),
                dtype=float,
            )
            outer_permutation_p = float(
                (1 + np.sum(null_aucs >= combined_auc))
                / (args.n_outer_permutations + 1)
            )
            outer_null_95 = float(np.quantile(null_aucs, 0.95))

            null_csv = result_dir / "nested_pipeline_auc_null.csv"
            with null_csv.open("w", newline="", encoding="utf-8") as handle:
                writer = csv.DictWriter(
                    handle, fieldnames=["permutation", "null_combined_oof_auc"]
                )
                writer.writeheader()
                writer.writerows(
                    {
                        "permutation": index,
                        "null_combined_oof_auc": value,
                    }
                    for index, value in enumerate(null_aucs, start=1)
                )

            null_figure, null_axis = plt.subplots(figsize=(7, 4))
            null_axis.hist(null_aucs, bins=30, color="#9AA7B1", edgecolor="white")
            null_axis.axvline(
                combined_auc,
                color="#A33A35",
                linewidth=2,
                label=f"Observed OOF AUC = {combined_auc:.3f}",
            )
            null_axis.axvline(
                outer_null_95,
                color="#204F78",
                linestyle="--",
                label=f"95% null = {outer_null_95:.3f}",
            )
            null_axis.set_xlabel("Combined out-of-fold AUC under permuted image labels")
            null_axis.set_ylabel("Count")
            null_axis.set_title(
                f"{SUBJECT} | {roi} | full nested-pipeline permutation test\n"
                f"p = {outer_permutation_p:.4g}"
            )
            null_axis.legend()
            null_figure.tight_layout()
            null_figure.savefig(
                result_dir / "nested_pipeline_auc_permutation_test.png", dpi=160
            )
            plt.close(null_figure)

        summary = {
            "subject": SUBJECT,
            "roi": roi,
            "feature_file": str(args.bin_file),
            "feature_method": feature_method,
            "outer_test_excluded_from_broadband_fit": strict_outer_preprocessing,
            "n_electrodes": len(roi_channels),
            "electrodes": roi_channels.tolist(),
            "n_permutations": args.n_permutations,
            "n_outer_pipeline_permutations": args.n_outer_permutations,
            "all_trials_have_outer_predictions": complete,
            "combined_out_of-fold_auc": combined_auc,
            "combined_oof_auc_permutation_p": outer_permutation_p,
            "null_auc_95th_percentile": outer_null_95,
            "warning": (
                "Pilot only: use at least 1000 full nested-pipeline permutations "
                "before confirmatory inference."
                if args.n_outer_permutations < 1000
                else None
            ),
        }
        (result_dir / "run_summary.json").write_text(
            json.dumps(summary, indent=2), encoding="utf-8"
        )
        print(json.dumps(summary, indent=2))
        print("Results:", result_dir)


In [ ]:
nested_decode = main
old_argv = sys.argv[:]
try:
    sys.argv = [
        "notebook",
        "--bin-file", str(STRICT_BIN_FILE),
        "--roi", ROI,
        "--n-permutations", str(INNER_PERMUTATIONS),
        "--n-outer-permutations", str(FULL_PIPELINE_PERMUTATIONS),
        "--n-jobs", str(N_JOBS),
        "--seed", str(SEED),
    ]
    nested_decode()
finally:
    sys.argv = old_argv


## 10. Results

The fold table reports windows chosen without each corresponding test fold and its independent
test AUC. The summary contains one combined out-of-fold AUC across all 300 predictions and
the full nested-pipeline permutation p-value.


In [ ]:
def read_csv_rows(path):
    with Path(path).open(newline="", encoding="utf-8-sig") as handle:
        return list(csv.DictReader(handle))

def display_rows(rows):
    if not rows:
        return
    columns = list(rows[0])
    head = "".join(f"<th>{column}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{row[column]}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

result_name = "nested_strict_" + "_".join(
    part for part in "".join(c if c.isalnum() else " " for c in ROI).lower().split()
)
RESULT_DIR = OUTPUT_DIR / result_name
summary = json.loads((RESULT_DIR / "run_summary.json").read_text(encoding="utf-8"))
display(summary)
display_rows(read_csv_rows(RESULT_DIR / "outer_fold_results.csv"))

for filename in (
    "inner_cv_cluster_selection.png",
    "held_out_test_roc.png",
    "nested_pipeline_auc_permutation_test.png",
):
    path = RESULT_DIR / filename
    print(filename)
    display(Image(filename=str(path)))


## Interpretation boundary

An out-of-fold AUC above 0.5 means that this participant's ROI ranks unseen face trials above
unseen house trials better than chance. The full-pipeline permutation p-value accounts for the
training-only temporal search. This remains a participant-level result: group inference must
repeat the same pipeline for the other participants and treat participants, not folds or
electrodes, as independent observations.
